Pydantic model of the ARLEM specification of an AR education activity.
Full spec, including some complications and extra fields that may not be needed in simpler education scenarios

In [ ]:
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field, HttpUrl


In [ ]:

# ==========================================
# PART 1: WORKPLACE MODEL (The Environment)
# ==========================================

class POI(BaseModel):
    """
    Defines a specific point of interest on a physical object relative to its center/origin.
    """
    id: str = Field(..., description="Unique identifier for this specific point.")
    x_offset: float = Field(0.0, description="Offset on the x-axis in cm.")
    y_offset: float = Field(0.0, description="Offset on the y-axis in cm.")
    z_offset: float = Field(0.0, description="Offset on the z-axis in cm.")
    x_rotation: float = Field(0.0, description="Pitch in Euler angles (degrees).")
    y_rotation: float = Field(0.0, description="Yaw in Euler angles (degrees).")
    z_rotation: float = Field(0.0, description="Roll in Euler angles (degrees).")

class Detectable(BaseModel):
    """
    Configuration for the computer vision system to recognize objects.
    """
    id: str = Field(..., description="Unique identifier referenced by Things, Places, or Persons.")
    type: Literal["marker", "anchor"] = Field(..., description="Type of tracking method used.")
    sensor: Optional[Literal["tracking", "mapping"]] = Field("tracking", description="The sensor subsystem to use (e.g., standard tracking or spatial mapping).")
    url: Optional[HttpUrl] = Field(None, description="URL to an asset bundle defining the marker or anchor data.")

class Tangible(BaseModel):
    """
    A physical object in the workplace.
    """
    id: str = Field(..., description="Unique identifier for the object.")
    name: str = Field(..., description="Human-readable name of the object.")
    detectable: Optional[str] = Field(None, description="ID of the Detectable used to track this object.")
    pois: List[POI] = Field(default_factory=list, description="List of Points of Interest defined on this object.")

class Person(Tangible):
    """
    A human user or instructor in the workplace.
    """
    twitter: Optional[str] = Field(None, description="Twitter handle for the person.")
    mbox: Optional[str] = Field(None, description="Email or ID used for xAPI logging identification.")
    persona: Optional[str] = Field(None, description="Role or group membership (e.g., 'learner', 'instructor').")

class SensorData(BaseModel):
    """
    Definition of a data stream variable.
    """
    key: str = Field(..., description="The variable identifier/topic in the data stream.")
    type: Literal["string", "float", "integer", "boolean"] = Field("string", description="The data type of the variable.")

class Sensor(BaseModel):
    """
    IoT sensor configuration for receiving external data.
    """
    id: str = Field(..., description="Unique identifier for the sensor.")
    url: str = Field(..., description="Connection URL (e.g., mqtt://host:port).")
    username: Optional[str] = Field(None, description="Auth username for the sensor stream.")
    password: Optional[str] = Field(None, description="Auth password for the sensor stream.")
    data: List[SensorData] = Field(default_factory=list, description="List of data variables to listen for.")

class Device(BaseModel):
    """
    Hardware available for the experience (e.g., HoloLens, Tablet).
    """
    id: str = Field(..., description="Unique identifier for the device.")
    type: str = Field(..., description="Type of device (e.g., 'hololens', 'tablet').")
    name: str = Field(..., description="Human-readable name of the device.")
    owner: Optional[str] = Field(None, description="Person ID of the device owner.")
    topic: Optional[str] = Field(None, description="MQTT channel name this device listens to.")

class App(BaseModel):
    """
    External web apps or widgets.
    """
    id: str = Field(..., description="Unique identifier for the app.")
    type: Literal["widget", "app", "prefab"] = Field(..., description="Type of application.")
    name: str = Field(..., description="Human-readable name.")
    url: str = Field(..., description="URL for launch command or download.")

class AugmentationPrimitive(BaseModel):
    """
    Base configuration for audio, video, models, or text.
    """
    id: str = Field(..., description="Unique identifier for this primitive.")
    type: Literal["animation", "image", "video", "audio", "label"] = Field(..., description="The media type of the augmentation.")
    url: Optional[HttpUrl] = Field(None, description="Source URL for the file (GLB, MP4, PNG, etc.).")
    scale: Optional[float] = Field(1.0, description="Normalized scaling factor.")
    # Optional size fields for bounding boxes
    x_size: Optional[float] = Field(None, description="Size on X axis (cm).")
    y_size: Optional[float] = Field(None, description="Size on Y axis (cm).")
    z_size: Optional[float] = Field(None, description="Size on Z axis (cm).")

class WarningSign(AugmentationPrimitive):
    """
    ISO 7010 Hazard signs.
    """
    symbol: Optional[str] = Field(None, description="Name of the standard ISO symbol.")

class Workplace(BaseModel):
    """
    The static environment definition containing all available resources.
    """
    id: str = Field(..., description="Unique identifier for this workplace environment.")
    name: str = Field(..., description="Descriptive name of the workplace.")
    origin: str = Field(..., description="ID of the Detectable that serves as the World Origin (0,0,0).")
    things: List[Tangible] = Field(default_factory=list, description="Physical tools, machines, or materials.")
    places: List[Tangible] = Field(default_factory=list, description="Locations or zones in the workplace.")
    persons: List[Person] = Field(default_factory=list, description="Users involved in the scenario.")
    sensors: List[Sensor] = Field(default_factory=list, description="Connected IoT devices.")
    devices: List[Device] = Field(default_factory=list, description="AR hardware used for delivery.")
    apps: List[App] = Field(default_factory=list, description="External widgets or apps.")
    detectables: List[Detectable] = Field(default_factory=list, description="Markers or Anchors used for tracking.")
    primitives: List[AugmentationPrimitive] = Field(default_factory=list, description="Available media assets (models, sounds).")
    predicates: List[AugmentationPrimitive] = Field(default_factory=list, description="Reusable instructional verbs (e.g., 'Rotate', 'Inspect').")
    warnings: List[WarningSign] = Field(default_factory=list, description="Standard safety signs.")


# ==========================================
# PART 2: ACTIVITY MODEL (The Logic)
# ==========================================

class Instruction(BaseModel):
    """
    Text/Audio instructions displayed to the user.
    """
    title: str = Field(..., description="Short headline of the instruction.")
    description: str = Field(..., description="Detailed narrative text.")

class Activate(BaseModel):
    """
    Command to spawn an object or effect.
    """
    target: str = Field(..., description="ID of the Thing, Place, Person, or Action to apply this to.")
    type: Literal["primitive", "predicate", "warning", "action"] = Field(..., description="Category of the object being activated.")
    augmentation: Optional[str] = Field(None, description="ID of the specific augmentation (from Workplace) to display.")
    poi: Optional[str] = Field(None, description="ID of the POI on the target where the augmentation appears.")
    url: Optional[HttpUrl] = Field(None, description="Override URL for dynamic content loading.")
    text: Optional[str] = Field(None, description="Content for text label primitives.")
    state: Optional[str] = Field(None, description="Keyframe or state ID for animations.")
    viewport: Optional[Literal["actions", "reactions", "warnings"]] = Field(None, description="If set, attaches object to the screen/HUD instead of the world.")
    option: Optional[str] = Field(None, description="Configuration option (e.g. direction 'up' for a pointer).")

class Deactivate(BaseModel):
    """
    Command to remove an object or effect.
    """
    target: str = Field(..., description="ID of the target to clear, or '*' for all.")
    type: Optional[Literal["primitive", "predicate", "warning", "action"]] = Field(None, description="Filter by type to remove.")
    augmentation: Optional[str] = Field(None, description="Specific augmentation ID to remove.")

class Message(BaseModel):
    """
    Communication between devices or users.
    """
    target: str = Field(..., description="ID of the Person or Device receiving the message.")
    type: Literal["person", "device", "sensor"] = Field(..., description="Type of recipient.")
    text: Optional[str] = Field(None, description="Body of the message.")
    key: Optional[str] = Field(None, description="Topic key for sensor/device broadcasting.")
    launch: Optional[str] = Field(None, description="ID of an action to trigger on the recipient device.")

class IfLogic(BaseModel):
    """
    Conditional logic based on xAPI statements (analytics).
    """
    url: str = Field(..., description="Query URL to check for xAPI statements.")
    then_action: str = Field(..., alias="then", description="Action ID to trigger if condition is met.")
    else_action: str = Field(..., alias="else", description="Action ID to trigger if condition is NOT met.")
    min_results: Optional[int] = Field(None, alias="min", description="Minimum number of statements required.")
    max_results: Optional[int] = Field(None, alias="max", description="Maximum number of statements allowed.")

class Trigger(BaseModel):
    """
    Event listener that causes the transition to the next step.
    """
    mode: Literal["click", "voice", "detect", "sensor"] = Field(..., description="Input method (UI click, voice command, object detection, or IoT data).")
    id: str = Field(..., description="ID of the entity to listen to (Action ID, Tangible ID, or Sensor ID).")
    duration: Optional[int] = Field(None, description="Time in ms required for gaze/detection to trigger.")
    key: Optional[str] = Field(None, description="Variable key to check (for sensor mode).")
    value: Optional[str] = Field(None, description="Threshold value to check (for sensor mode).")
    operator: Optional[Literal["equal", "exceed", "below", "between"]] = Field("equal", description="Comparison operator (for sensor mode).")

class ActionFlow(BaseModel):
    """
    The set of commands executed when entering or exiting a step.
    """
    remove_self: bool = Field(False, alias="removeSelf", description="If True, removes the current instruction/UI immediately.")
    activate: List[Activate] = Field(default_factory=list, description="List of items to display.")
    deactivate: List[Deactivate] = Field(default_factory=list, description="List of items to hide.")
    messages: List[Message] = Field(default_factory=list, description="Messages to send.")
    if_logic: List[IfLogic] = Field(default_factory=list, alias="if", description="Conditional logic checks.")

class Action(BaseModel):
    """
    A single step in the activity workflow.
    """
    id: str = Field(..., description="Unique ID for this step.")
    instruction: Optional[Instruction] = Field(None, description="Text displayed to the user during this step.")
    enter: Optional[ActionFlow] = Field(None, description="Commands executed immediately when this step starts.")
    exit: Optional[ActionFlow] = Field(None, description="Commands executed when the Trigger conditions are met.")
    triggers: List[Trigger] = Field(default_factory=list, description="Conditions that cause the step to complete.")

class Activity(BaseModel):
    """
    The root object defining the logic flow of the AR experience.
    """
    id: str = Field(..., description="Unique ID for the activity.")
    name: str = Field(..., description="Human-readable title.")
    description: Optional[str] = Field(None, description="Overview of the activity.")
    language: str = Field("en", min_length=2, max_length=2, description="ISO 639-1 language code.")
    workplace: Union[HttpUrl, str] = Field(..., description="URL or Reference to the Workplace definition file.")
    start: str = Field(..., description="ID of the first Action to execute.")
    actions: List[Action] = Field(default_factory=list, description="All available steps/states in this activity.")